In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

In [2]:
class SimpleState(TypedDict):
    proposed_action:str
    approvel:bool
    finish : str

In [4]:
def proposed_action(state : SimpleState)->dict:
    action = 'proposel for the first message'
    print(f'[propesing_action] -->{action}')
    return {'proposed_action':{action}}



In [6]:
def human_approvol(state : SimpleState)->dict:
    print("need human approvol")

    answer = interrupt({
        'request':f'approv this action {state['proposed_action']}',
        'hint':"reply yes or no"
    })

    approved = (str(answer).strip().lower() == "yes")
    print(f"[ask_human] Resumed! Human said: '{answer}' --> approved={approved}")
    return {"approved": approved}

In [7]:
def finish(state : SimpleState)->dict:
    if state['approved']:
        result=f'human said ok EXCUET'
    else :
        f'human didnt accept the request'
    return {'result':result}

In [9]:
builder = StateGraph(SimpleState)

In [10]:
builder.add_node('proposed_action',proposed_action)
builder.add_node('approvel',human_approvol)
builder.add_node('finish',finish)


In [11]:
builder.add_edge(START,'proposed_action')
builder.add_edge('proposed_action','approvel')
builder.add_edge('approvel','finish')
builder.add_edge('finish',END)

In [12]:
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [13]:

print("Graph compiled. Starting first invoke...\n")

config = {"configurable": {"thread_id": "demo-thread-1"}}

result = graph.invoke(
    {"proposed_action": "", "approved": False, "result": ""},
    config
)
print(f"\nFirst invoke returned: {result}")

Graph compiled. Starting first invoke...

[propesing_action] -->proposel for the first message
need human approvol

First invoke returned: {'proposed_action': {'proposel for the first message'}, '__interrupt__': [Interrupt(value={'request': "approv this action {'proposel for the first message'}", 'hint': 'reply yes or no'}, id='6d53f5a64ce5085551fed29dfc8b4643')]}
